# Phase 01 — Zenodo source audit

**Project:** Open Materials Data Pipeline for Hydrogel Bioink Design
**Source:** [Zenodo record 19602891](https://zenodo.org/records/19602891) · DOI [10.5281/zenodo.19602891](https://doi.org/10.5281/zenodo.19602891)

**What this notebook does.** It inventories and quality-reviews the two source files before any schema is designed. It confirms the files are the audited release, checks the ZIP archive's integrity, profiles every worksheet and column, records unit variants and missing values in the image index, and lists candidate filename matches between the two rheology folders. Every finding goes into an issue register for Phase 02. The source files are opened read-only; nothing is extracted, converted or changed.

| | |
|---|---|
| **Inputs** | `data/raw/zenodo_19602891/ALG-Ph_HA-Ph_rheology_data.zip`; `data/raw/zenodo_19602891/image_index.csv` |
| **Outputs** | 11 audit tables (CSV) and `audit_run_manifest.json` in `data/metadata/zenodo_source_audit/`; `phase01_checks.csv` in the same folder |
| **Runtime** | About 20 seconds |
| **Requires** | Phase 00 passed (environment and source files verified) |

**How to run.** Select the **Python (hydrogel-bioink)** kernel (or the kernel built from `environment.yml`), then **Kernel → Restart Kernel and Run All Cells**. The summary in section 9 reports every check as **PASS**, **WARN** or **FAIL**; a failure stops the notebook with the reason. Section 10 lists the findings with the status labels defined in Phase 00: **Verified**, **Preliminary**, **Assumption**, **Unresolved**.

**Scope.** This is an inventory and quality review. Counts of rows are counts of measurement points within worksheets, not counts of independent samples. The audit does not evaluate printability, fit rheological models or confirm which files describe the same physical sample.

## 1. Settings

Fixed values used throughout the notebook: input locations, the reference MD5 checksums of the audited release, the counts recorded by the published audit, and the SHA-256 checksums of its 11 output tables. The reference values make the audit testable: a rerun on the same inputs is expected to reproduce them exactly.

In [1]:
from collections import Counter
from datetime import datetime, timezone
from importlib import metadata
from pathlib import Path, PurePosixPath
import csv
import hashlib
import io
import json
import platform
import re
import zipfile

import numpy as np
import openpyxl
import pandas as pd

pd.set_option("display.max_colwidth", None)

RAW_RELATIVE = Path("data/raw/zenodo_19602891")
OUT_RELATIVE = Path("data/metadata/zenodo_source_audit")
SOURCE_RECORD = "https://zenodo.org/records/19602891"

REFERENCE_MD5 = {   # published checksums of the audited release (v0.0.2, reviewed 2 October 2026)
    "ALG-Ph_HA-Ph_rheology_data.zip": "4eaac901afc46212108bfdad8b39b5c2",
    "image_index.csv": "e0fc1a61b6f5d5b2ea87a69a97d4aa58",
}
EXPECTED = {        # counts recorded by the published Phase 01 audit
    "workbooks": 92, "workbooks_per_folder": {"GG_data": 46, "viscosity_data": 46},
    "worksheets": 92, "measurement_rows": 4600, "distinct_headers": 13,
    "image_records": 1031, "candidate_stems": 62, "one_to_one_stems": 30, "issues": 15,
}
REFERENCE_EXPORT_SHA256 = {   # SHA-256 of each output table from the published audit
    "source_manifest.csv": "5fefea8090bbd598e12020f5b1be93eccde87a7a4b83faaa7836a5f54fa48be7",
    "archive_member_inventory.csv": "a82ea7580b4941869ca0c1b2698efd50e7e4feec57678cfd0497237d0eb50751",
    "workbook_structure_inventory.csv": "e546d31e4f0434a9e797ee3555ca71b25afb653162dbacfc793a3ee813f1ee9a",
    "source_header_inventory.csv": "8184bfff6a5d13b587044bfd0098f266c78542fb4aad30e6c1a1637d8f3b0ed7",
    "measurement_column_profile.csv": "120aa04a12e56993d3821dc82be5569af30fe074039f509023f3fd05c236dca8",
    "measurement_axis_profile.csv": "c0a30820727068b30ab3ca7d8a2aa49df353e5599982c2a5c07ddcf3c85bc40c",
    "image_index_column_profile.csv": "5885ff29927a70921059954aacacba7eba8ff33bcbad253337e99979a0459a60",
    "image_index_value_counts.csv": "e2fbe1568b7d0043c5750f6fbfa05fdb52fccd34afde73b1af978db879460666",  # new explicit sort order (see section 6)
    "candidate_filename_correspondences.csv": "a3f95d08e9393b7e871d9a2103b85f2c0306bb46357d36b8ac03cd6496e4003e",
    "image_filename_expectations.csv": "6f0aff02ad2afb05ae39c53ff9da7313fbabd033f392253769d5d4acbb3ab759",
    "audit_issue_log.csv": "1b898b430040b1ea0e5cc76362923d3f7bc1d830e49d5eb60ef8223da4f6013e",
}

checks = []   # validation results: one row per check
issues = []   # audit findings: one row per issue, exported as audit_issue_log.csv

def record(check_id, description, passed, detail, warn_only=False):
    status = "PASS" if passed else ("WARN" if warn_only else "FAIL")
    checks.append({"check_id": check_id, "description": description, "status": status, "detail": detail})
    print(f"[{status:4}] {check_id}: {detail}")

def stop_if_failed(check_id):
    # Stop immediately when a check that later steps depend on has failed.
    if checks[-1]["check_id"] == check_id and checks[-1]["status"] == "FAIL":
        raise RuntimeError(f"{check_id} failed: {checks[-1]['detail']}")

def flag(code, source, detail, action, severity="review", sheet="", column=""):
    issues.append(dict(issue_code=code, severity=severity, source=source,
                       sheet=sheet, column=column, evidence=detail, next_action=action))

def digest(path, algorithm):
    h = hashlib.new(algorithm)
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

RUN_UTC = datetime.now(timezone.utc).isoformat()
environment = pd.DataFrame([{"component": "Python", "version": platform.python_version()}] +
                           [{"component": p, "version": metadata.version(p)} for p in ("pandas", "numpy", "openpyxl", "ipython")])
print("Settings loaded. Python", platform.python_version(), "| pandas", pd.__version__, "| numpy", np.__version__)

Settings loaded. Python 3.12.16 | pandas 2.2.3 | numpy 2.5.3


## 2. Locate the project and verify the source files

The project folder is found the same way as in Phase 00. Both source files must exist under their original names and match the reference MD5 checksums. A missing or different file stops the notebook: an audit of the wrong file would produce valid-looking but misleading tables.

The source manifest records each file's size, MD5 and SHA-256. The dataset's reuse licence is logged as unresolved because the record page did not state a dataset-specific licence when reviewed.

In [2]:
here = Path.cwd().resolve()
project_root = next((p for p in (here, *here.parents)
                     if (p / "README.md").is_file() and (p / "environment.yml").is_file() and (p / "notebooks").is_dir()), None)
if project_root is None:
    raise FileNotFoundError("Project folder not found. Open this notebook from inside the hydrogel-bioink-data-curation repository.")
raw_dir = project_root / RAW_RELATIVE
output_dir = project_root / OUT_RELATIVE
record("P01-01", "Project folder located", True, f"repository root found ('{project_root.name}')")

missing = [name for name in REFERENCE_MD5 if not (raw_dir / name).is_file()]
record("P01-02", "Source files present", not missing,
       "both files present" if not missing else f"missing in {RAW_RELATIVE.as_posix()}/: {', '.join(missing)} — run: python scripts/download_data.py")
stop_if_failed("P01-02")
zip_path, index_path = (raw_dir / name for name in REFERENCE_MD5)

manifest_rows = []
for canonical, path in zip(REFERENCE_MD5, (zip_path, index_path)):
    actual = digest(path, "md5")
    manifest_rows.append(dict(source_record=SOURCE_RECORD, source_version="v0.0.2", metadata_review_date="2026-10-02",
        canonical_filename=canonical, selected_filename=path.name,
        relative_path=path.relative_to(project_root).as_posix(), bytes=path.stat().st_size,
        md5=actual, reference_md5=REFERENCE_MD5[canonical], checksum_match=actual == REFERENCE_MD5[canonical],
        sha256=digest(path, "sha256"), dataset_licence_status="unresolved"))
source_manifest = pd.DataFrame(manifest_rows)
mismatched = source_manifest.loc[~source_manifest.checksum_match, "canonical_filename"].tolist()
record("P01-03", "Source files match the audited release (MD5)", not mismatched,
       "both checksums match" if not mismatched else f"checksum differs: {', '.join(mismatched)} — re-download with scripts/download_data.py")
stop_if_failed("P01-03")

flag("DATASET_LICENCE_UNRESOLVED", "Zenodo record 19602891",
     "Dataset licence identifier not established in this audit.",
     "Confirm dataset reuse and redistribution terms from authoritative metadata.")
source_manifest[["canonical_filename", "bytes", "checksum_match", "dataset_licence_status"]]

[PASS] P01-01: repository root found ('hydrogel-github-main')
[PASS] P01-02: both files present
[PASS] P01-03: both checksums match


,canonical_filename,bytes,checksum_match,dataset_licence_status
0,ALG-Ph_HA-Ph_rheology_data.zip,1121259,True,unresolved
1,image_index.csv,37937,True,unresolved


## 3. Archive integrity and file inventory

The ZIP archive is checked before any workbook is read: no member may have an unsafe path (absolute, `..` or a drive letter), no name may repeat, sizes must stay within set limits, and every member must pass its CRC check. Each file's size and SHA-256 are recorded, which also reveals byte-identical duplicates. Nothing is extracted to disk.

*Scientific note:* folder names (`GG_data`, `viscosity_data`) suggest the test type, but the test protocol is established from the measured variables in section 5, not from the folder name.

In [3]:
MAX_MEMBER_BYTES = 50 * 1024 * 1024
MAX_ARCHIVE_EXPANDED_BYTES = 250 * 1024 * 1024
with zipfile.ZipFile(zip_path) as archive:
    members = archive.infolist()
    unsafe = [m.filename for m in members if PurePosixPath(m.filename).is_absolute()
              or ".." in PurePosixPath(m.filename).parts or "\\" in m.filename or re.match(r"^[A-Za-z]:", m.filename)]
    repeated = [name for name, n in Counter(m.filename for m in members).items() if n > 1]
    oversized = (any(m.file_size > MAX_MEMBER_BYTES for m in members)
                 or sum(m.file_size for m in members) > MAX_ARCHIVE_EXPANDED_BYTES)
    bad_member = archive.testzip()
    problems = ([f"unsafe paths {unsafe[:3]}"] if unsafe else []) + ([f"repeated names {repeated[:3]}"] if repeated else []) \
             + (["size limits exceeded"] if oversized else []) + ([f"CRC failure in {bad_member}"] if bad_member else [])
    record("P01-04", "Archive is safe and intact", not problems,
           f"{len(members)} members; paths safe, names unique, CRC passed" if not problems else "; ".join(problems))
    stop_if_failed("P01-04")
    archive_inventory = pd.DataFrame([dict(member_path=m.filename, is_directory=m.is_dir(),
        folder=PurePosixPath(m.filename).parent.as_posix(), suffix=PurePosixPath(m.filename).suffix.lower(),
        bytes=m.file_size, compressed_bytes=m.compress_size, crc32=f"{m.CRC:08x}",
        sha256="" if m.is_dir() else hashlib.sha256(archive.read(m)).hexdigest()) for m in members])

files = archive_inventory.loc[~archive_inventory.is_directory].copy()
xlsx_members = sorted(files.loc[files.suffix.eq(".xlsx"), "member_path"].tolist())
for _, row in files.loc[files.duplicated("sha256", keep=False)].iterrows():
    flag("DUPLICATE_MEMBER_CONTENT", row.member_path, "SHA-256 matches another archived file.",
         "Determine whether this is a duplicated export or a documented repeated record.")
for _, row in files.loc[~files.suffix.eq(".xlsx")].iterrows():
    flag("UNPARSED_ARCHIVE_MEMBER", row.member_path, "Non-XLSX file retained in inventory.",
         "Review format and documentation relevance; excluded from workbook parsing.")

per_folder = files.loc[files.suffix.eq(".xlsx")].groupby("folder").size().to_dict()
record("P01-05", "Workbook count matches the published audit", per_folder == EXPECTED["workbooks_per_folder"],
       f"{len(xlsx_members)} workbooks: " + ", ".join(f"{k} {v}" for k, v in sorted(per_folder.items())))
stop_if_failed("P01-05")

[PASS] P01-04: 94 members; paths safe, names unique, CRC passed
[PASS] P01-05: 92 workbooks: GG_data 46, viscosity_data 46


## 4. Worksheet structure and column profiles

Every worksheet is read twice from memory: once for cell values and once for formulas, which are listed but never evaluated. Row 1 is taken as the header row. For each column the profile records the exact header, its position, blank cells, how many values are numeric, and the minimum, median and maximum. Formula cells, Excel error cells, blank rows, repeated rows and non-positive rheology values are logged as issues with their source coordinates. No value is changed or removed.

*Scientific note:* G′ (storage modulus) and G″ (loss modulus) describe elastic and viscous response under oscillation; η is steady-shear viscosity and |η\*| is complex viscosity. These are different observables and are kept separate. Units stay attached to each header: strain as a fraction (`ɣ in -`) and as a percentage (`ɣ in %`) differ by a factor of 100, and the `mPas` viscosity label is preserved as written.

In [4]:
def is_blank(value):
    return value is None or (isinstance(value, str) and not value.strip())

def text_value(value):
    return "" if value is None else str(value)

sheet_rows, column_rows, axis_rows = [], [], []
for member in xlsx_members:
    with zipfile.ZipFile(zip_path) as archive:
        payload = archive.read(member)
    try:
        wb_values = openpyxl.load_workbook(io.BytesIO(payload), read_only=True, data_only=True)
        wb_formulas = openpyxl.load_workbook(io.BytesIO(payload), read_only=True, data_only=False)
    except Exception as exc:
        flag("WORKBOOK_READ_ERROR", member, f"{type(exc).__name__}: {exc}",
             "Inspect this workbook; audit coverage is incomplete.", severity="error")
        continue
    try:
        for ws in wb_values:
            rows = list(ws.iter_rows(values_only=True))
            formula_cells = list(wb_formulas[ws.title].iter_rows())
            if not rows:
                flag("EMPTY_SHEET", member, "No readable cells.", "Review sheet purpose.", sheet=ws.title)
                continue
            width = max(len(r) for r in rows)
            headers = [text_value(v) for v in rows[0]] + [""] * (width - len(rows[0]))
            data = [(i, list(r) + [None] * (width - len(r))) for i, r in enumerate(rows[1:], 2)]
            active = [(i, r) for i, r in data if not all(is_blank(v) for v in r)]
            blank_rows = [i for i, r in data if all(is_blank(v) for v in r)]
            formulas = [c.coordinate for row in formula_cells for c in row if c.data_type == "f"]
            excel_errors = [c.coordinate for row in formula_cells for c in row if c.data_type == "e"]
            if any(not h.strip() for h in headers) or len(set(headers)) != len(headers):
                flag("HEADER_REVIEW", member, "Blank or duplicate candidate row-1 headers.",
                     "Confirm header row and use column positions before schema mapping.", sheet=ws.title)
            for issue_code, values, action in [
                ("FORMULA_CELLS", formulas, "Check cached results; formulas are not evaluated by this reader."),
                ("EXCEL_ERROR_CELLS", excel_errors, "Inspect original cells and instrument export."),
                ("BLANK_DATA_ROWS", blank_rows, "Check whether blanks separate acquisition segments.")]:
                if values:
                    flag(issue_code, member, json.dumps(values), action, sheet=ws.title)
            seen, duplicate_rows = set(), []
            for i, row in active:
                signature = tuple((type(v).__name__, str(v)) for v in row)
                if signature in seen:
                    duplicate_rows.append(i)
                seen.add(signature)
            if duplicate_rows:
                flag("DUPLICATE_MEASUREMENT_ROWS", member, json.dumps(duplicate_rows),
                     "Review acquisition/export context; do not automatically delete.", sheet=ws.title)
            sheet_rows.append(dict(member_path=member, sheet=ws.title, reported_max_row=ws.max_row,
                reported_max_column=ws.max_column, candidate_header_row=1, data_rows=len(active),
                blank_data_rows=len(blank_rows), duplicate_rows=len(duplicate_rows),
                formula_cells=len(formulas), excel_error_cells=len(excel_errors),
                headers_json=json.dumps(headers, ensure_ascii=False)))
            for j, header in enumerate(headers):
                vals = [r[j] for _, r in active]
                blank = [is_blank(v) for v in vals]
                numeric = pd.to_numeric(pd.Series([None if b else v for v, b in zip(vals, blank)], dtype=object), errors="coerce")
                finite = numeric.notna() & np.isfinite(numeric.astype(float))
                clean = numeric[finite].astype(float)
                position = openpyxl.utils.get_column_letter(j + 1)
                col = dict(member_path=member, sheet=ws.title, column_position=j + 1, excel_column=position,
                    source_header=header, observed_rows=len(vals), blank_count=sum(blank),
                    blank_percent=100 * sum(blank) / len(vals) if vals else 0,
                    numeric_count=int(finite.sum()),
                    nonnumeric_nonblank_count=sum(not b and pd.isna(n) for b, n in zip(blank, numeric)),
                    nonfinite_numeric_count=int((numeric.notna() & ~finite).sum()),
                    distinct_nonblank=len({text_value(v) for v, b in zip(vals, blank) if not b}),
                    numeric_min=clean.min() if len(clean) else None,
                    numeric_max=clean.max() if len(clean) else None,
                    numeric_median=clean.median() if len(clean) else None)
                column_rows.append(col)
                if col["blank_count"]:
                    flag("BLANK_MEASUREMENT_CELLS", member, f"{sum(blank)} of {len(vals)} cells blank; column {position}.",
                         "Review missingness before extraction or imputation.", sheet=ws.title, column=header)
                if " in " in header and (col["nonnumeric_nonblank_count"] or col["nonfinite_numeric_count"]):
                    flag("NUMERIC_FIELD_REVIEW", member, f"Non-numeric: {col['nonnumeric_nonblank_count']}; nonfinite: {col['nonfinite_numeric_count']}.",
                         "Review source cell values and missing-value conventions.", sheet=ws.title, column=header)
                if header in ["G' in Pa", 'G" in Pa', "η in mPas", "|η*| in mPas"]:
                    affected = [i for (i, _), v in zip(active, numeric) if pd.notna(v) and np.isfinite(v) and v <= 0]
                    if affected:
                        flag("NONPOSITIVE_RHEOLOGY_VALUE", member, "Excel rows: " + json.dumps(affected),
                             "Review measurement range and instrument limits; retain values pending review.", sheet=ws.title, column=header)
                if header in ["ɣ in -", "ɣ in %", "f in Hz", "ɣ̇ in 1/s", "T in °C"]:
                    axis_rows.append({**{k: col[k] for k in ["member_path", "sheet", "source_header", "numeric_min", "numeric_max"]},
                                      "distinct_numeric_values": int(clean.nunique())})
    finally:
        wb_values.close()
        wb_formulas.close()

workbook_inventory = pd.DataFrame(sheet_rows)
column_profile = pd.DataFrame(column_rows)
axis_profile = pd.DataFrame(axis_rows)
header_inventory = (column_profile.groupby("source_header", dropna=False)
                    .agg(occurrences=("source_header", "size"), blank_cells=("blank_count", "sum")).reset_index())

read_errors = sum(i["issue_code"] == "WORKBOOK_READ_ERROR" for i in issues)
record("P01-06", "All workbooks readable", read_errors == 0 and workbook_inventory.member_path.nunique() == len(xlsx_members),
       f"{workbook_inventory.member_path.nunique()} of {len(xlsx_members)} workbooks read; {read_errors} read errors")
stop_if_failed("P01-06")
n_rows = int(workbook_inventory.data_rows.sum())
record("P01-07", "Worksheet and row counts match the published audit",
       len(workbook_inventory) == EXPECTED["worksheets"] and n_rows == EXPECTED["measurement_rows"],
       f"{len(workbook_inventory)} worksheets, {n_rows:,} measurement rows")
record("P01-08", "Header vocabulary matches the published audit", len(header_inventory) == EXPECTED["distinct_headers"],
       f"{len(header_inventory)} distinct headers")
header_inventory

[PASS] P01-06: 92 of 92 workbooks read; 0 read errors
[PASS] P01-07: 92 worksheets, 4,600 measurement rows
[PASS] P01-08: 13 distinct headers


,source_header,occurrences,blank_cells
0,"G"" in Pa",46,0
1,G' in Pa,46,0
2,SegIndex,92,0
3,T in °C,92,0
4,f in Hz,46,0
5,t in s,92,0
6,t_seg in s,92,0
7,|η*| in mPas,46,0
8,ɣ in %,2,0
9,ɣ in -,44,0


## 5. Unit variants and acquisition axes

Two strain headers occur: `ɣ in -` (fraction) and `ɣ in %` (percentage). Both are kept as reported; the conversion rule is defined in Phase 03. The axis profile counts how many distinct values each test variable takes within a worksheet, which shows what was varied during the test.

*Scientific note:* in all 46 modulus worksheets the frequency is fixed at 1 Hz while the strain varies, which is the pattern of an amplitude sweep rather than a frequency sweep. A linear viscoelastic region is therefore not assumed. In the viscosity worksheets the shear rate varies, consistent with a flow curve.

In [5]:
strain_headers = set(column_profile.source_header) & {"ɣ in -", "ɣ in %"}
if len(strain_headers) > 1:
    flag("MIXED_STRAIN_UNITS", "rheology archive", "Both fraction and percentage strain headers occur.",
         "Preserve source strain units; define and validate conversion during schema standardization.")
strain_counts = column_profile.loc[column_profile.source_header.isin(strain_headers)].groupby("source_header").size()
frequency = axis_profile.loc[axis_profile.source_header.eq("f in Hz")]
print("Strain headers:", ", ".join(f"'{h}' in {n} worksheets" for h, n in strain_counts.items()))
print(f"Frequency axis: {len(frequency)} modulus worksheets; distinct values per worksheet = "
      f"{sorted(int(v) for v in frequency.distinct_numeric_values.unique())}; range {frequency.numeric_min.min():g}–{frequency.numeric_max.max():g} Hz")
column_profile.loc[column_profile.source_header.eq("ɣ in %"), ["member_path", "source_header", "numeric_min", "numeric_max"]]

Strain headers: 'ɣ in %' in 2 worksheets, 'ɣ in -' in 44 worksheets
Frequency axis: 46 modulus worksheets; distinct values per worksheet = [1]; range 1–1 Hz


,member_path,source_header,numeric_min,numeric_max
317,GG_data/ALG-I1G-Ph_4.0_HA-HQ-Ph_3.0_22.5C_0.xlsx,ɣ in %,0.011359,22.229651
367,GG_data/ALG-I1G-Ph_5.0_HA-HQ-Ph_2.0_22.5C_1.xlsx,ɣ in %,0.005846,31.597120


## 6. Image index: structure and missing values

The CSV is read as text so identifiers keep their exact form and blank fields stay blank. The checks cover column names, row widths, unique image IDs, date format and intensity values. Blank annotations are counted and recorded as unknown; a blank is not read as "no" or as a failed print.

The value-count table is sorted by column, then by count (highest first), then by value. This explicit order makes the table identical on every computer; the original version left tied counts in an order that varied between machines.

*Scientific note:* the intensity column has no unit in its header, so it is recorded as unresolved. `completeness_sam` is described in the source record as incomplete and unused in the original study.

In [6]:
with index_path.open("r", encoding="utf-8-sig", newline="") as stream:
    csv_rows = list(csv.reader(stream))
headers = csv_rows[0] if csv_rows else []
malformed = [i for i, r in enumerate(csv_rows[1:], 2) if len(r) != len(headers)]
structure_ok = bool(csv_rows) and all(h.strip() for h in headers) and len(set(headers)) == len(headers) and not malformed
record("P01-09", "Image index readable with consistent row widths", structure_ok,
       f"{len(csv_rows) - 1} records x {len(headers)} columns" if structure_ok else f"empty, blank/duplicate headers, or uneven rows {malformed[:10]}")
stop_if_failed("P01-09")

image_index = pd.DataFrame(csv_rows[1:], columns=headers)
expected_columns = ["image_id", "date", "ink_type_ph", "shape", "intensity", "completeness_sam", "contain_sps"]
if headers != expected_columns:
    flag("IMAGE_SCHEMA_DIFFERENCE", index_path.name, json.dumps(headers), "Review header order, added fields and missing expected columns.")
missing_mask = image_index.apply(lambda s: s.str.strip().eq(""))
image_profile = pd.DataFrame({
    "column": headers, "rows": len(image_index),
    "blank_count": [int(missing_mask[c].sum()) for c in headers],
    "blank_percent": [float(missing_mask[c].mean() * 100) for c in headers],
    "distinct_nonblank": [int(image_index.loc[~missing_mask[c], c].nunique()) for c in headers],
})
for row in image_profile.itertuples():
    if row.blank_count:
        flag("IMAGE_METADATA_BLANK", index_path.name, f"{row.blank_count} of {row.rows} records blank ({row.blank_percent:.1f}%).",
             "Retain as unknown; investigate collection/annotation coverage.", column=row.column)

def record_numbers(mask):
    return (image_index.index[mask] + 2).tolist()

dup_rows = image_index.duplicated(keep=False)
if dup_rows.any():
    flag("DUPLICATE_IMAGE_RECORD", index_path.name, json.dumps(record_numbers(dup_rows)), "Check whether records were duplicated during index assembly.")
duplicate_id = image_index.image_id.duplicated(keep=False) & ~missing_mask.image_id
if duplicate_id.any():
    flag("NONUNIQUE_IMAGE_ID", index_path.name, json.dumps(record_numbers(duplicate_id)), "Resolve identifier granularity before joins.", column="image_id")
bad_dates = pd.to_datetime(image_index.date, format="%Y/%m/%d", errors="coerce").isna() & ~missing_mask.date
if bad_dates.any():
    flag("DATE_PARSE_REVIEW", index_path.name, json.dumps(record_numbers(bad_dates)), "Check source date convention; preserve raw text.", column="date")
intensity_numeric = pd.to_numeric(image_index.intensity, errors="coerce")
bad_intensity = (~np.isfinite(intensity_numeric) | (intensity_numeric < 0)) & ~missing_mask.intensity
if bad_intensity.any():
    flag("INTENSITY_VALUE_REVIEW", index_path.name, json.dumps(record_numbers(bad_intensity)), "Confirm value conventions and units.", column="intensity")
flag("INTENSITY_UNIT_UNRESOLVED", index_path.name, "CSV header does not specify intensity units.",
     "Check experimental methods before conversion or dose calculations.", column="intensity")

column_order = {c: i for i, c in enumerate(headers)}
image_value_counts = pd.DataFrame(
    [dict(column=c, source_value=value, is_blank=not value.strip(), count=int(count))
     for c in headers if c != "image_id" for value, count in image_index[c].value_counts().items()])
image_value_counts = (image_value_counts.assign(_order=image_value_counts.column.map(column_order))
                      .sort_values(["_order", "count", "source_value"], ascending=[True, False, True], kind="stable")
                      .drop(columns="_order").reset_index(drop=True))

record("P01-10", "Image index matches the published audit",
       len(image_index) == EXPECTED["image_records"] and headers == expected_columns and not duplicate_id.any(),
       f"{len(image_index):,} records, expected columns, image_id unique: {not duplicate_id.any()}")
image_profile

[PASS] P01-09: 1031 records x 7 columns
[PASS] P01-10: 1,031 records, expected columns, image_id unique: True


,column,rows,blank_count,blank_percent,distinct_nonblank
0,image_id,1031,0,0.000000,1031
1,date,1031,0,0.000000,18
2,ink_type_ph,1031,0,0.000000,48
3,shape,1031,0,0.000000,4
4,intensity,1031,0,0.000000,7
5,completeness_sam,1031,943,91.464597,1
6,contain_sps,1031,922,89.427740,1


## 7. Candidate filename correspondences

Workbooks in the two folders are paired when their file names (without extension) are identical. Each pair is a **candidate** correspondence: a shared name suggests the two tests used the same formulation, but it does not confirm the same physical sample, batch or run. Every pair is therefore marked `physical_sample_link = unresolved`. No fuzzy matching is applied.

The expected image file name for each index record is also listed (`resized_IMG_<id>.png`); whether the images exist is tested in Phase 06.

In [7]:
member_groups = {}
for member in xlsx_members:
    p = PurePosixPath(member)
    member_groups.setdefault(p.stem, {}).setdefault(p.parent.as_posix(), []).append(member)
filename_correspondences = pd.DataFrame([dict(source_stem=stem,
        GG_count=len(groups.get("GG_data", [])), viscosity_count=len(groups.get("viscosity_data", [])),
        GG_members=json.dumps(groups.get("GG_data", [])), viscosity_members=json.dumps(groups.get("viscosity_data", [])),
        exact_one_each=len(groups.get("GG_data", [])) == 1 and len(groups.get("viscosity_data", [])) == 1,
        physical_sample_link="unresolved") for stem, groups in sorted(member_groups.items())])
flag("PHYSICAL_SAMPLE_LINK_UNRESOLVED", "cross-source linkage",
     "Filename correspondences are not verified sample or replicate identifiers.",
     "Obtain a source-supported formulation/batch/sample/run crosswalk before merging.")
flag("IMAGE_FILES_NOT_AUDITED", "images.zip", "Image archive is outside the selected inputs.",
     "Check indexed filenames against the image archive in a later audit.")
image_file_expectations = pd.DataFrame({
    "source_csv_record": image_index.index + 2, "image_id": image_index.image_id,
    "expected_filename": image_index.image_id.map(lambda v: f"resized_IMG_{v}.png" if v.strip() else ""),
    "existence_status": "not tested: image archive not supplied"})

n_stems, n_pairs = len(filename_correspondences), int(filename_correspondences.exact_one_each.sum())
record("P01-11", "Candidate correspondences match the published audit",
       n_stems == EXPECTED["candidate_stems"] and n_pairs == EXPECTED["one_to_one_stems"],
       f"{n_stems} file-name stems; {n_pairs} with exactly one workbook in each folder")

[PASS] P01-11: 62 file-name stems; 30 with exactly one workbook in each folder


## 8. Save the audit tables

The 11 tables and the run manifest are written to `data/metadata/zenodo_source_audit/`, replacing any earlier run. Before writing, the source files are re-hashed to confirm they were not changed during the audit. After writing, each table's SHA-256 is compared with the published audit. A difference is reported as a WARN: the tables are still valid, but the difference should be explained (for example, a different package version) before downstream phases rely on them.

| Table | Contents |
|---|---|
| `source_manifest.csv` | Source files with size, MD5, SHA-256 and licence status |
| `archive_member_inventory.csv` | Every ZIP member with size, CRC and SHA-256 |
| `workbook_structure_inventory.csv` | One row per worksheet: rows, blanks, formulas, headers |
| `source_header_inventory.csv` | Each distinct header and how often it occurs |
| `measurement_column_profile.csv` | One row per worksheet column: blanks, numeric coverage, min/median/max |
| `measurement_axis_profile.csv` | Strain, frequency, shear-rate and temperature ranges per worksheet |
| `image_index_column_profile.csv` | Blanks and distinct values per image-index column |
| `image_index_value_counts.csv` | Every value in each image-index column with its count |
| `candidate_filename_correspondences.csv` | File-name pairs between the two rheology folders |
| `image_filename_expectations.csv` | Expected image file name per index record |
| `audit_issue_log.csv` | Every finding with its evidence and recommended handling |

In [8]:
earlier_failures = [f"{c['check_id']}: {c['detail']}" for c in checks if c["status"] == "FAIL"]
if earlier_failures:
    raise RuntimeError("Outputs not written because earlier checks failed:\n  " + "\n  ".join(earlier_failures))

issue_columns = ["issue_code", "severity", "source", "sheet", "column", "evidence", "next_action"]
issue_log = pd.DataFrame(issues, columns=issue_columns).sort_values(
    ["issue_code", "source", "sheet", "column"], kind="stable").reset_index(drop=True)
issue_log.insert(0, "issue_id", [f"AUDIT-{i:04d}" for i in range(1, len(issue_log) + 1)])

unchanged = all(digest(p, "sha256") == s for p, s in zip((zip_path, index_path), source_manifest.sha256))
record("P01-12", "Source files unchanged during the audit", unchanged,
       "SHA-256 identical before and after" if unchanged else "a source file changed during the run; outputs not written")
stop_if_failed("P01-12")

exports = {
    "source_manifest.csv": source_manifest, "archive_member_inventory.csv": archive_inventory,
    "workbook_structure_inventory.csv": workbook_inventory, "source_header_inventory.csv": header_inventory,
    "measurement_column_profile.csv": column_profile, "measurement_axis_profile.csv": axis_profile,
    "image_index_column_profile.csv": image_profile, "image_index_value_counts.csv": image_value_counts,
    "candidate_filename_correspondences.csv": filename_correspondences,
    "image_filename_expectations.csv": image_file_expectations, "audit_issue_log.csv": issue_log,
}
output_dir.mkdir(parents=True, exist_ok=True)
for name, frame in exports.items():
    frame.to_csv(output_dir / name, index=False, encoding="utf-8-sig", lineterminator="\n")
export_sha256 = {name: digest(output_dir / name, "sha256") for name in exports}

summary = {
    "run_utc": RUN_UTC, "source_record": SOURCE_RECORD, "source_metadata_review_date": "2026-10-02",
    "environment": environment.to_dict(orient="records"),
    "xlsx_files_in_archive": len(xlsx_members), "xlsx_files_audited": int(workbook_inventory.member_path.nunique()),
    "worksheets_audited": len(workbook_inventory), "measurement_rows": n_rows,
    "image_index_rows": len(image_index), "issues": len(issue_log),
    "input_sha256": dict(zip(source_manifest.canonical_filename, source_manifest.sha256)),
    "export_sha256": export_sha256,
    "dataset_licence": "unresolved", "sample_linkage": "unresolved", "image_existence": "not tested", "raw_files_preserved": True,
}
(output_dir / "audit_run_manifest.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")

record("P01-13", "Issue count matches the published audit", len(issue_log) == EXPECTED["issues"], f"{len(issue_log)} findings logged")
differing = [name for name, sha in export_sha256.items() if sha != REFERENCE_EXPORT_SHA256[name]]
record("P01-14", "Audit tables identical to the published audit (SHA-256)", not differing,
       f"{len(exports) - len(differing)} of {len(exports)} tables identical" + (f"; differ: {', '.join(differing)}" if differing else ""),
       warn_only=True)
print(f"Saved {len(exports)} tables and audit_run_manifest.json to {OUT_RELATIVE.as_posix()}/")

[PASS] P01-12: SHA-256 identical before and after
[PASS] P01-13: 15 findings logged
[PASS] P01-14: 11 of 11 tables identical
Saved 11 tables and audit_run_manifest.json to data/metadata/zenodo_source_audit/


## 9. Validation summary

Every check from this run, followed by the overall result. A failed check stops the notebook here, so a failed run cannot be mistaken for a successful one. The check table is also saved as `phase01_checks.csv`.

In [9]:
check_table = pd.DataFrame(checks)
check_table.to_csv(output_dir / "phase01_checks.csv", index=False, encoding="utf-8-sig", lineterminator="\n")
counts = check_table.status.value_counts()
n_pass, n_warn, n_fail = (int(counts.get(s, 0)) for s in ("PASS", "WARN", "FAIL"))
display(check_table)
summary_line = f"Phase 01: {n_pass} of {len(check_table)} checks passed, {n_warn} warning(s), {n_fail} failure(s)."
if n_fail:
    raise RuntimeError(summary_line + "\n" + "\n".join(f"  - {r.check_id}: {r.detail}" for r in check_table[check_table.status == "FAIL"].itertuples()))
print(summary_line)

,check_id,description,status,detail
0,P01-01,Project folder located,PASS,repository root found ('hydrogel-github-main')
1,P01-02,Source files present,PASS,both files present
2,P01-03,Source files match the audited release (MD5),PASS,both checksums match
3,P01-04,Archive is safe and intact,PASS,"94 members; paths safe, names unique, CRC passed"
4,P01-05,Workbook count matches the published audit,PASS,"92 workbooks: GG_data 46, viscosity_data 46"
5,P01-06,All workbooks readable,PASS,92 of 92 workbooks read; 0 read errors
6,P01-07,Worksheet and row counts match the published audit,PASS,"92 worksheets, 4,600 measurement rows"
7,P01-08,Header vocabulary matches the published audit,PASS,13 distinct headers
8,P01-09,Image index readable with consistent row widths,PASS,1031 records x 7 columns
9,P01-10,Image index matches the published audit,PASS,"1,031 records, expected columns, image_id unique: True"


Phase 01: 14 of 14 checks passed, 0 warning(s), 0 failure(s).


## 10. Findings

Each row summarises one group of findings from the issue log, with its status label, what it means, and whether it needs attention. Full evidence, including Excel row numbers, is in `audit_issue_log.csv`.

In [10]:
def issue_count(code):
    return int((issue_log.issue_code == code).sum())

blank_cols = image_profile.loc[image_profile.blank_count > 0]
nonpositive = issue_log.loc[issue_log.issue_code.eq("NONPOSITIVE_RHEOLOGY_VALUE")]
structural = ["FORMULA_CELLS", "EXCEL_ERROR_CELLS", "BLANK_DATA_ROWS", "BLANK_MEASUREMENT_CELLS",
              "DUPLICATE_MEASUREMENT_ROWS", "HEADER_REVIEW", "DUPLICATE_MEMBER_CONTENT", "UNPARSED_ARCHIVE_MEMBER"]
structural_found = {c: issue_count(c) for c in structural if issue_count(c)}
fixed_frequency = bool(len(frequency)) and frequency.distinct_numeric_values.eq(1).all()
findings = pd.DataFrame([
    {"finding": f"Source archive intact: {len(xlsx_members)} workbooks, {len(workbook_inventory)} worksheets, {n_rows:,} measurement rows",
     "status": "Verified",
     "meaning": ("No formulas, Excel errors, blank cells, repeated rows, bad headers or duplicate files were found" if not structural_found
                 else "Structural issues found: " + ", ".join(f"{c} ({n})" for c, n in structural_found.items())),
     "attention": "No" if not structural_found else "Yes — see audit_issue_log.csv"},
    {"finding": f"Measurement rows are repeated points within {len(workbook_inventory)} worksheets",
     "status": "Verified", "meaning": f"The {n_rows:,} rows are not {n_rows:,} independent samples; any model must group by worksheet",
     "attention": "No action in Phase 01"},
    {"finding": f"Mixed strain units: 'ɣ in -' in {strain_counts.get('ɣ in -', 0)} worksheets, 'ɣ in %' in {strain_counts.get('ɣ in %', 0)}",
     "status": "Verified", "meaning": "Comparing them without a rule would introduce a factor-of-100 error",
     "attention": "Yes — conversion rule defined in Phase 03"},
    {"finding": f"Non-positive G′, G″, η or |η*| values in {nonpositive.source.nunique()} worksheets ({issue_count('NONPOSITIVE_RHEOLOGY_VALUE')} columns)",
     "status": "Preliminary", "meaning": "Physically unusual; possibly near the instrument's measurement limit. Values retained with their Excel rows",
     "attention": "Yes — review before analysis"},
    {"finding": (f"All {len(frequency)} modulus worksheets vary strain at a single fixed frequency" if fixed_frequency
                 else "Modulus worksheets do not all use a single fixed frequency"),
     "status": "Assumption" if fixed_frequency else "Unresolved",
     "meaning": ("Treated as amplitude sweeps from the measured variables; protocol documentation not yet confirmed" if fixed_frequency
                 else "Test type cannot be assigned from the measured variables alone"),
     "attention": "No" if fixed_frequency else "Yes — review the axis profile"},
    {"finding": "Blank image annotations: " + "; ".join(f"{r.column} {r.blank_count} of {r.rows}" for r in blank_cols.itertuples()),
     "status": "Verified", "meaning": "Blanks are recorded as unknown, not as 'no' or failed prints",
     "attention": "No"},
    {"finding": "Intensity unit not stated in the image index",
     "status": "Unresolved", "meaning": "Values kept as source tokens; no dose or energy calculation is possible",
     "attention": "Yes — check the publication's methods"},
    {"finding": f"{n_stems} workbook name stems; {n_pairs} with one modulus and one viscosity file",
     "status": "Unresolved", "meaning": "Name matches are candidates only; physical-sample identity is not confirmed",
     "attention": "No action in Phase 01; Phase 04 tests linkage"},
    {"finding": "Image files not checked against the index",
     "status": "Unresolved", "meaning": "Expected file names listed; existence tested in Phase 06",
     "attention": "No action in Phase 01"},
    {"finding": "Dataset reuse and redistribution terms",
     "status": "Unresolved", "meaning": "Raw files are not committed to the repository",
     "attention": "Yes — confirm the licence on the Zenodo record"},
])
findings

,finding,status,meaning,attention
0,"Source archive intact: 92 workbooks, 92 worksheets, 4,600 measurement rows",Verified,"No formulas, Excel errors, blank cells, repeated rows, bad headers or duplicate files were found",No
1,Measurement rows are repeated points within 92 worksheets,Verified,"The 4,600 rows are not 4,600 independent samples; any model must group by worksheet",No action in Phase 01
2,"Mixed strain units: 'ɣ in -' in 44 worksheets, 'ɣ in %' in 2",Verified,Comparing them without a rule would introduce a factor-of-100 error,Yes — conversion rule defined in Phase 03
3,"Non-positive G′, G″, η or |η*| values in 8 worksheets (8 columns)",Preliminary,Physically unusual; possibly near the instrument's measurement limit. Values retained with their Excel rows,Yes — review before analysis
4,All 46 modulus worksheets vary strain at a single fixed frequency,Assumption,Treated as amplitude sweeps from the measured variables; protocol documentation not yet confirmed,No
5,Blank image annotations: completeness_sam 943 of 1031; contain_sps 922 of 1031,Verified,"Blanks are recorded as unknown, not as 'no' or failed prints",No
6,Intensity unit not stated in the image index,Unresolved,Values kept as source tokens; no dose or energy calculation is possible,Yes — check the publication's methods
7,62 workbook name stems; 30 with one modulus and one viscosity file,Unresolved,Name matches are candidates only; physical-sample identity is not confirmed,No action in Phase 01; Phase 04 tests linkage
8,Image files not checked against the index,Unresolved,Expected file names listed; existence tested in Phase 06,No action in Phase 01
9,Dataset reuse and redistribution terms,Unresolved,Raw files are not committed to the repository,Yes — confirm the licence on the Zenodo record


**Handoff to Phase 02.** The audit tables in `data/metadata/zenodo_source_audit/` describe exactly what the source contains: headers and units per worksheet, measurement axes, image-index vocabularies and every finding with its source coordinates. Phase 02 uses them to design the schema, data dictionary and quality-control rules. Phase 02 reads `audit_run_manifest.json` and re-verifies every table against the SHA-256 values recorded there.

**References.** [Zenodo record](https://zenodo.org/records/19602891) (version and checksums reviewed 2 October 2026) · [Associated publication](https://doi.org/10.1080/17452759.2026.2671497) · [Authors' code](https://github.com/KORINZ/generative-ai-bioprinting-framework)